# 01 Data Access and Selection / Sampling

This notebook implements the first two stages of the reproducible workflow for the Europeana India metadata project:

1. Data access
2. Selection / sampling

The project investigates how European cultural heritage institutions represent India through metadata in Europeana records.

The dataset was collected from the Europeana API in May 2026 using keyword-based queries related to India. The records were organised into four query groups:

- Geographic
- Colonial
- Cultural
- Material

The purpose of this notebook is to load the collected dataset, inspect its structure, preserve a raw copy, and create a selected analytical subset for later cleaning and analysis.

In [1]:
# Import required libraries
import pandas as pd
from pathlib import Path

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
from pathlib import Path

current_folder = Path.cwd()

if current_folder.name == "notebooks":
    project_dir = current_folder.parent
else:
    project_dir = current_folder

data_dir = project_dir / "data"
raw_dir = data_dir / "raw"
processed_dir = data_dir / "processed"

raw_dir.mkdir(parents=True, exist_ok=True)
processed_dir.mkdir(parents=True, exist_ok=True)

print("Current working folder:", current_folder)
print("Project folder:", project_dir)
print("Raw data folder:", raw_dir)
print("Processed data folder:", processed_dir)

Current working folder: C:\Users\kevin\Documents\GitHub\Open_Project_2026\notebooks
Project folder: C:\Users\kevin\Documents\GitHub\Open_Project_2026
Raw data folder: C:\Users\kevin\Documents\GitHub\Open_Project_2026\data\raw
Processed data folder: C:\Users\kevin\Documents\GitHub\Open_Project_2026\data\processed


In [3]:
# Load the preserved raw Europeana dataset

input_file = raw_dir / "europeana_india_dataset_1500_raw.csv"

if not input_file.exists():
    raise FileNotFoundError(
        f"Raw dataset not found: {input_file}"
    )

df = pd.read_csv(input_file)

print("Dataset loaded successfully.")
print("Input file:", input_file)
print("Shape:", df.shape)

df.head()

Dataset loaded successfully.
Input file: C:\Users\kevin\Documents\GitHub\Open_Project_2026\data\raw\europeana_india_dataset_1500_raw.csv
Shape: (1500, 8)


,query_group,search_term,title,description,subject,dataProvider,type,country
0,geographic,India,India,"Barev.\n28 x 36 cm, na listu 38 x 43 cm\nMěřít...",NaN,Map Collection UK,IMAGE,['Czech Republic']
1,geographic,India,India,NaN,NaN,Internet Culturale,SOUND,['Italy']
2,geographic,India,India,NaN,NaN,Internet Culturale,SOUND,['Italy']
3,geographic,India,India,"Characters and interpreters: Banjo; Frisell, B...",NaN,Internet Culturale,SOUND,['Italy']
4,geographic,India,India.,"Litografie, kolor.\n28,5 x 37 cm na listu 31 x...",NaN,Map Collection UK,IMAGE,['Czech Republic']


## Inspecting the dataset

This step checks which metadata fields are available in the dataset. These fields are needed to understand whether the dataset can answer the research question.

In [4]:
# Show available columns

print("Columns in the dataset:")
print(df.columns.tolist())

Columns in the dataset:
['query_group', 'search_term', 'title', 'description', 'subject', 'dataProvider', 'type', 'country']


## Selection and sampling

The dataset was collected using four thematic query groups. This step checks how many records belong to each query group. This helps evaluate whether the selection strategy captured different dimensions of India-related cultural heritage metadata.

In [5]:
# Check distribution of query groups

print("Query group distribution:")
print(df["query_group"].value_counts())

Query group distribution:
query_group
geographic    375
colonial      375
cultural      375
material      375
Name: count, dtype: int64


## Creating a deduplicated analytical subset

Initial exploration showed that the dataset contains repeated titles. This can happen because Europeana may include multiple records for volumes, copies, or similar collection objects.

For later analysis, a deduplicated subset is created using the `title` column. The original raw dataset is not changed.

In [6]:
# Create deduplicated analytical subset based on title

df_unique = df.drop_duplicates(subset="title").copy()

print("Original dataset shape:", df.shape)
print("Deduplicated dataset shape:", df_unique.shape)

df_unique.head()

Original dataset shape: (1500, 8)
Deduplicated dataset shape: (245, 8)


,query_group,search_term,title,description,subject,dataProvider,type,country
0,geographic,India,India,"Barev.\n28 x 36 cm, na listu 38 x 43 cm\nMěřít...",NaN,Map Collection UK,IMAGE,['Czech Republic']
4,geographic,India,India.,"Litografie, kolor.\n28,5 x 37 cm na listu 31 x...",NaN,Map Collection UK,IMAGE,['Czech Republic']
10,geographic,India,%india%,NaN,NaN,Royal Museums Greenwich,IMAGE,['United Kingdom']
16,geographic,India,India & Nepál,"Tartalom ◊ I. ◊ Az első randevú: Delhi, 1972 ◊...",NaN,National Széchényi Library of Hungary,TEXT,['Hungary']
18,geographic,India,India SONG,NaN,NaN,National Audiovisual Institute France,SOUND,['France']


In [7]:
# Save deduplicated dataset for later analysis

processed_output_file = processed_dir / "europeana_india_unique_titles.csv"

df_unique.to_csv(processed_output_file, index=False, encoding="utf-8")

print("Deduplicated analytical dataset saved to:")
print(processed_output_file)

Deduplicated analytical dataset saved to:
C:\Users\kevin\Documents\GitHub\Open_Project_2026\data\processed\europeana_india_unique_titles.csv


## Reproducibility check

This final step checks whether the expected output files were created successfully.

In [8]:
# Check that the input and output files exist

print("Raw input file exists:", input_file.exists())
print("Processed output file exists:", processed_output_file.exists())

print("\nRaw input file:")
print(input_file)

print("\nProcessed output file:")
print(processed_output_file)

print("\nFiles in data/raw:")
print([file.name for file in raw_dir.iterdir()])

print("\nFiles in data/processed:")
print([file.name for file in processed_dir.iterdir()])

Raw input file exists: True
Processed output file exists: True

Raw input file:
C:\Users\kevin\Documents\GitHub\Open_Project_2026\data\raw\europeana_india_dataset_1500_raw.csv

Processed output file:
C:\Users\kevin\Documents\GitHub\Open_Project_2026\data\processed\europeana_india_unique_titles.csv

Files in data/raw:
['europeana_india_dataset_1500_raw.csv']

Files in data/processed:
['europeana_india_unique_titles.csv', 'europeana_india_unique_titles_enhanced_sample.csv', 'europeana_india_unique_titles_enhanced_top15_providers.csv', 'europeana_india_unique_titles_transformed.csv']


## Summary of this notebook

This notebook implemented the first two stages of the project workflow: data access and selection / sampling.

The Europeana India metadata dataset was loaded from CSV, inspected, and saved as a raw copy in `data/raw/`. The selection strategy was checked by counting records in each query group. The result showed a balanced dataset with 375 records in each of the four query groups: geographic, colonial, cultural, and material.

A deduplicated analytical subset was then created using unique titles and saved in `data/processed/`. The raw dataset is preserved for transparency, while the deduplicated subset will be used for further cleaning, analysis, and visualisation.